In [4]:
pip install fastapi uvicorn langchain langchain-community faiss-cpu openai sentence-transformers python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
documents = [

    {
        "id": "doc1",
        "title": "Cloud Computing",
        "text": "Cloud computing provides scalable computing resources over the internet.",
        "category": "technology",
        "date": "2024-01-10"
    },

    {
        "id": "doc2",
        "title": "Machine Learning",
        "text": "Machine learning enables systems to learn patterns from data.",
        "category": "technology",
        "date": "2024-03-12"
    },

    {
        "id": "doc3",
        "title": "Travel Guide",
        "text": "Travel planning helps tourists optimize routes and budgets.",
        "category": "travel",
        "date": "2023-11-01"
    }

]

In [6]:
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

/tmp/ipykernel_6010/3311835203.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:
langchain_docs = []

for doc in documents:

    langchain_docs.append(
        Document(
            page_content=doc["text"],
            metadata={
                "id": doc["id"],
                "title": doc["title"],
                "category": doc["category"],
                "date": doc["date"]
            }
        )
    )

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_6010/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
vectorstore = FAISS.from_documents(
    langchain_docs,
    embedding_model
)

In [10]:
def retrieve(query, top_k=3):

    results = vectorstore.similarity_search_with_score(
        query,
        k=top_k
    )

    retrieved_docs = []

    scores = []

    for doc, score in results:

        similarity = 1 / (1 + score)

        retrieved_docs.append({
            "text": doc.page_content,
            "title": doc.metadata["title"],
            "id": doc.metadata["id"]
        })

        scores.append(similarity)

    max_score = max(scores)

    return retrieved_docs, max_score

In [11]:
def generate_answer(query):

    docs, confidence = retrieve(query)

    context = "\n".join([
        doc["text"] for doc in docs
    ])

    sources = [
        doc["title"] for doc in docs
    ]

    answer = f"""
Answer based on retrieved documents:

{context}

Question:
{query}
"""

    if confidence < 0.3:

        answer += "\n\nWARNING: Low confidence retrieval."

    return {
        "answer": answer,
        "sources": sources,
        "confidence": round(confidence, 3)
    }